# Scenario Execution Report

Professional run report for the Adaptive Quantum SOR scenario API. This notebook resets the scenario boundary, submits the configured parent order set, runs the scenario, and presents a clean report of what ran, what happened, and what the result was.

Run the cells from top to bottom. The final cell is the shippable report view.

## 1. Report Inputs

Use the widget panel in the next cell to choose the scenario, reset mode, order side, urgency, quantity, and route limits. Keep `reset_mode = 'PURGE_AND_REPOPULATE'` when you want a clean, replay-friendly user report. The execution cell reads the current widget values when it runs.

In [ ]:
from datetime import datetime, timezone
from html import escape
from IPython.display import HTML, display
import importlib
import pandas as pd

from adaptive_quantum_sor import find_scenario, load_scenarios
import adaptive_quantum_sor.client as sor_client_module

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    widgets = None

sor_client_module = importlib.reload(sor_client_module)
SorNotebookClient = sor_client_module.SorNotebookClient

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

client = SorNotebookClient()

scenarios = load_scenarios()
scenario_lookup = {scenario.scenario_id: scenario for scenario in scenarios}

DEFAULT_SCENARIO_ID = "quadratic-pair-override"
DEFAULT_SEED = int(scenario_lookup.get(DEFAULT_SCENARIO_ID, next(iter(scenario_lookup.values()))).seed)
DEFAULT_TICKS = int(scenario_lookup.get(DEFAULT_SCENARIO_ID, next(iter(scenario_lookup.values()))).ticks)

scenario_id = DEFAULT_SCENARIO_ID
seed = DEFAULT_SEED
ticks = DEFAULT_TICKS
reset_mode = "PURGE_AND_REPOPULATE"
max_route_attempts = 8
route_timeout_millis = 1000
parent_order_count = 2
instrument_id = 0
side = "BUY"
quantity = 7000
urgency_id = 1
at_tick = 1
submit_mode = "SIMULATED"
client_order_ref = "scenario-report-order"

def build_parent_orders(count, instrument_id, side, quantity, urgency_id, at_tick, submit_mode, client_order_ref):
    rows = []
    for index in range(int(count)):
        rows.append({
            "instrumentId": int(instrument_id),
            "side": side,
            "quantity": int(quantity),
            "urgencyId": int(urgency_id),
            "atTick": int(at_tick) + index,
            "submitMode": submit_mode,
            "clientOrderRef": f"{client_order_ref}-{index + 1}",
        })
    return rows

parent_orders = build_parent_orders(parent_order_count, instrument_id, side, quantity, urgency_id, at_tick, submit_mode, client_order_ref)

widget_controls = {}
if widgets is not None:
    scenario_options = [(f"{scenario.scenario_id} ({scenario.category})", scenario.scenario_id) for scenario in scenarios]
    widget_controls = {
        "scenario_id": widgets.Dropdown(options=scenario_options, value=DEFAULT_SCENARIO_ID, description="Scenario", layout=widgets.Layout(width="520px")),
        "seed": widgets.IntText(value=DEFAULT_SEED, description="Seed"),
        "ticks": widgets.IntSlider(value=DEFAULT_TICKS, min=1, max=120, step=1, description="Ticks", continuous_update=False),
        "reset_mode": widgets.Dropdown(options=["PURGE_AND_REPOPULATE", "KEEP_POLICY_PURGE_STATS", "ISOLATED", "APPEND"], value="PURGE_AND_REPOPULATE", description="Reset"),
        "max_route_attempts": widgets.IntSlider(value=8, min=1, max=32, step=1, description="Attempts", continuous_update=False),
        "route_timeout_millis": widgets.IntSlider(value=1000, min=100, max=10000, step=100, description="Timeout ms", continuous_update=False),
        "parent_order_count": widgets.IntSlider(value=2, min=1, max=5, step=1, description="Orders", continuous_update=False),
        "instrument_id": widgets.IntSlider(value=0, min=0, max=1, step=1, description="Instrument", continuous_update=False),
        "side": widgets.Dropdown(options=["BUY", "SELL"], value="BUY", description="Side"),
        "quantity": widgets.IntText(value=7000, description="Quantity"),
        "urgency_id": widgets.Dropdown(options=[("LOW", 0), ("NORMAL", 1), ("HIGH", 2)], value=1, description="Urgency"),
        "at_tick": widgets.IntSlider(value=1, min=0, max=max(1, DEFAULT_TICKS - 1), step=1, description="At tick", continuous_update=False),
        "submit_mode": widgets.Dropdown(options=["SIMULATED", "API"], value="SIMULATED", description="Submit"),
        "client_order_ref": widgets.Text(value="scenario-report-order", description="Ref", layout=widgets.Layout(width="360px")),
    }

    def _sync_scenario_defaults(change):
        scenario = scenario_lookup[change["new"]]
        widget_controls["seed"].value = int(scenario.seed)
        widget_controls["ticks"].value = int(scenario.ticks)
        widget_controls["at_tick"].max = max(0, int(scenario.ticks) - 1)

    def _sync_tick_bounds(change):
        widget_controls["at_tick"].max = max(0, int(change["new"]) - 1)

    widget_controls["scenario_id"].observe(_sync_scenario_defaults, names="value")
    widget_controls["ticks"].observe(_sync_tick_bounds, names="value")

    display(HTML("<div class='sor-report sor-section'><h2>Widget Control Panel</h2><p>Choose the run setup here, then run the next cell to generate the report.</p></div>"))
    display(widgets.VBox([
        widgets.HBox([widget_controls["scenario_id"], widget_controls["reset_mode"]]),
        widgets.HBox([widget_controls["seed"], widget_controls["ticks"], widget_controls["max_route_attempts"], widget_controls["route_timeout_millis"]]),
        widgets.HBox([widget_controls["parent_order_count"], widget_controls["instrument_id"], widget_controls["side"], widget_controls["urgency_id"]]),
        widgets.HBox([widget_controls["quantity"], widget_controls["at_tick"], widget_controls["submit_mode"], widget_controls["client_order_ref"]]),
    ]))
else:
    display(HTML("<div class='sor-report sor-note'>ipywidgets is not installed. Install python/requirements.txt to enable dropdown controls; the notebook will use the Python defaults below.</div>"))

def read_widget_inputs():
    if widget_controls:
        selected_scenario_id = widget_controls["scenario_id"].value
        selected_seed = int(widget_controls["seed"].value)
        selected_ticks = int(widget_controls["ticks"].value)
        selected_reset_mode = widget_controls["reset_mode"].value
        selected_max_attempts = int(widget_controls["max_route_attempts"].value)
        selected_timeout = int(widget_controls["route_timeout_millis"].value)
        selected_orders = build_parent_orders(
            widget_controls["parent_order_count"].value,
            widget_controls["instrument_id"].value,
            widget_controls["side"].value,
            widget_controls["quantity"].value,
            widget_controls["urgency_id"].value,
            min(widget_controls["at_tick"].value, selected_ticks - 1),
            widget_controls["submit_mode"].value,
            widget_controls["client_order_ref"].value,
        )
        return selected_scenario_id, selected_seed, selected_ticks, selected_reset_mode, selected_max_attempts, selected_timeout, selected_orders
    return scenario_id, seed, ticks, reset_mode, max_route_attempts, route_timeout_millis, parent_orders

REPORT_CSS = """
<style>
:root {
  --sor-ink: #111827;
  --sor-muted: #5b6472;
  --sor-line: #d7dde6;
  --sor-panel: #f8fafc;
  --sor-panel-2: #eef5ff;
  --sor-blue: #1d4ed8;
  --sor-green: #047857;
  --sor-amber: #b45309;
  --sor-red: #b91c1c;
}
.sor-report {font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; color: var(--sor-ink);}
.sor-hero {border: 1px solid var(--sor-line); border-radius: 8px; padding: 18px 20px; margin: 4px 0 18px 0; background: linear-gradient(180deg, #ffffff 0%, #f8fafc 100%);}
.sor-hero h1 {font-size: 24px; margin: 0 0 6px 0; letter-spacing: 0;}
.sor-hero p {margin: 0; color: var(--sor-muted); font-size: 14px;}
.sor-section {margin: 22px 0 10px 0;}
.sor-section h2 {font-size: 18px; margin: 0 0 4px 0; letter-spacing: 0;}
.sor-section p {margin: 0 0 8px 0; color: var(--sor-muted); font-size: 13px;}
.sor-kpi-grid {display: grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap: 10px; margin: 10px 0 18px 0;}
.sor-kpi {border: 1px solid var(--sor-line); border-radius: 8px; padding: 12px 14px; background: #ffffff;}
.sor-kpi-label {font-size: 12px; color: var(--sor-muted); margin-bottom: 5px;}
.sor-kpi-value {font-size: 20px; font-weight: 700; line-height: 1.15;}
.sor-badge {display: inline-block; border-radius: 999px; padding: 3px 9px; font-size: 12px; font-weight: 700;}
.sor-pass {background: #ecfdf5; color: var(--sor-green); border: 1px solid #a7f3d0;}
.sor-warn {background: #fffbeb; color: var(--sor-amber); border: 1px solid #fde68a;}
.sor-fail {background: #fef2f2; color: var(--sor-red); border: 1px solid #fecaca;}
.sor-note {border-left: 4px solid var(--sor-blue); background: var(--sor-panel-2); padding: 10px 12px; margin: 10px 0 18px 0; color: #1e3a8a;}
table.sor-report-table {border-collapse: collapse; width: 100%; table-layout: auto; margin: 8px 0 18px 0; font-size: 13px;}
table.sor-report-table th, table.sor-report-table td {border: 1px solid var(--sor-line); padding: 7px 9px; vertical-align: top; white-space: normal; overflow-wrap: anywhere; word-break: break-word; max-width: 560px; text-align: left;}
table.sor-report-table th {background: var(--sor-panel); color: var(--sor-ink); font-weight: 700;}
table.sor-report-table tbody tr:nth-child(even) {background: #fcfdff;}
</style>
"""

def display_styles():
    display(HTML(REPORT_CSS))


def display_header(title, subtitle, generated_at):
    display(HTML(f"""
    <div class="sor-report sor-hero">
      <h1>{escape(title)}</h1>
      <p>{escape(subtitle)}</p>
      <p>Generated at {escape(generated_at)} UTC</p>
    </div>
    """))


def display_section(title, subtitle=""):
    subtitle_html = f"<p>{escape(subtitle)}</p>" if subtitle else ""
    display(HTML(f"<div class='sor-report sor-section'><h2>{escape(title)}</h2>{subtitle_html}</div>"))


def display_note(text):
    display(HTML(f"<div class='sor-report sor-note'>{escape(text)}</div>"))


def display_table(title, frame, subtitle=""):
    display_section(title, subtitle)
    if frame is None or frame.empty:
        display_note("No rows were returned for this section.")
        return
    html = frame.to_html(index=False, escape=True)
    html = html.replace('class="dataframe"', 'class="sor-report-table"')
    display(HTML("<div class='sor-report'>" + html + "</div>"))


def display_kpis(cards):
    html_cards = []
    for label, value in cards:
        html_cards.append(
            "<div class='sor-kpi'>"
            f"<div class='sor-kpi-label'>{escape(str(label))}</div>"
            f"<div class='sor-kpi-value'>{escape(str(value))}</div>"
            "</div>"
        )
    display(HTML("<div class='sor-report sor-kpi-grid'>" + "".join(html_cards) + "</div>"))


def badge(label, tone):
    css = {"pass": "sor-pass", "warn": "sor-warn", "fail": "sor-fail"}.get(tone, "sor-warn")
    return f"<span class='sor-badge {css}'>{escape(str(label))}</span>"


def frame_from_rows(rows, columns=None):
    frame = pd.DataFrame(rows)
    if columns:
        for column in columns:
            if column not in frame.columns:
                frame[column] = ""
        frame = frame[columns]
    return frame


def numeric_sum(frame, column):
    if frame is None or frame.empty or column not in frame.columns:
        return 0
    return int(pd.to_numeric(frame[column], errors="coerce").fillna(0).sum())


def format_bool(value):
    return "Yes" if bool(value) else "No"


def status_name(status):
    mapping = {0: "NEW", 1: "ACKED", 2: "PARTIALLY_FILLED", 3: "FILLED", 4: "REJECTED", 5: "CANCELLED"}
    try:
        return mapping.get(int(status), str(status))
    except (TypeError, ValueError):
        return str(status)


display_styles()
scenario_id, seed, ticks, reset_mode, max_route_attempts, route_timeout_millis, parent_orders = read_widget_inputs()
planned_orders = frame_from_rows(parent_orders, [
    "clientOrderRef", "instrumentId", "side", "quantity", "urgencyId", "atTick", "submitMode"
])
manifest = frame_from_rows([{
    "scenarioId": scenario_id,
    "seed": seed,
    "ticks": ticks,
    "resetMode": reset_mode,
    "maxRouteAttempts": max_route_attempts,
    "routeTimeoutMillis": route_timeout_millis,
    "parentOrderCount": len(parent_orders),
}])

display_header(
    "Scenario Execution Report - Planned Run",
    "Configuration preview before any API call is made.",
    datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
)
display_table("Run Manifest", manifest, "These are the scenario controls that will be sent to the SOR API.")
display_table("Parent Orders To Submit", planned_orders, "Each row is one parent order that will be routed during the scenario run.")


## 2. Execute And Generate Report

Run the next cell to execute the reset and scenario run. The report that appears below is designed to be read by a user without inspecting raw JSON.

In [ ]:
scenario_id, seed, ticks, reset_mode, max_route_attempts, route_timeout_millis, parent_orders = read_widget_inputs()
started_at = datetime.now(timezone.utc)
reset = client.reset_scenario(scenario_id, seed=seed, ticks=ticks, reset_mode=reset_mode)
run = client.run_scenario(
    scenario_id,
    seed=seed,
    ticks=ticks,
    reset_mode=reset_mode,
    parent_orders=parent_orders,
    max_route_attempts=max_route_attempts,
    route_timeout_millis=route_timeout_millis,
    simulator_generated_orders=False,
)
summary = client.scenario_summary_dataframe()
events = client.scenario_events_dataframe()
completed_at = datetime.now(timezone.utc)

parent_order_results = pd.DataFrame([
    {key: value for key, value in result.items() if key != "fills"}
    for result in run.get("parentOrderResults", [])
])
if not parent_order_results.empty and "status" in parent_order_results.columns:
    parent_order_results["statusName"] = parent_order_results["status"].map(status_name)

fill_rows = []
for result in run.get("parentOrderResults", []):
    for fill in result.get("fills", []):
        row = dict(fill)
        row["clientOrderRef"] = result.get("clientOrderRef", "")
        fill_rows.append(row)
fills = pd.DataFrame(fill_rows)

reset_frame = pd.DataFrame([reset]).add_prefix("reset.")
run_frame = pd.DataFrame([run]).drop(columns=["parentOrderResults"], errors="ignore").add_prefix("run.")
summary_frame = summary.add_prefix("summary.")

order_count = len(parent_orders)
completed_orders = 0 if parent_order_results.empty or "remainingQty" not in parent_order_results.columns else int((pd.to_numeric(parent_order_results["remainingQty"], errors="coerce").fillna(0) == 0).sum())
timed_out_orders = 0 if parent_order_results.empty or "timedOut" not in parent_order_results.columns else int(parent_order_results["timedOut"].astype(bool).sum())
total_filled = numeric_sum(parent_order_results, "filledQty")
total_remaining = numeric_sum(parent_order_results, "remainingQty")
child_order_count = numeric_sum(parent_order_results, "childOrderCount")
route_attempts = numeric_sum(parent_order_results, "routeAttempts")
venue_count = 0 if fills.empty or "venueId" not in fills.columns else int(fills["venueId"].nunique())
run_success = bool(run.get("success", False))
replay_safe = bool(run.get("replaySafe", False))
needs_review = (not run_success) or (not replay_safe) or timed_out_orders > 0 or total_remaining > 0
outcome_label = "Completed" if not needs_review else "Needs review"
outcome_tone = "pass" if not needs_review else "warn"

result_rows = [
    {"Check": "Run accepted by API", "Result": format_bool(run_success), "Meaning": run.get("message", "")},
    {"Check": "Replay-safe reset mode", "Result": format_bool(replay_safe), "Meaning": "Clean replay evidence" if replay_safe else "Append or non-replay-safe mode used"},
    {"Check": "Parent orders submitted", "Result": order_count, "Meaning": "Configured parent order count"},
    {"Check": "Parent orders fully filled", "Result": completed_orders, "Meaning": "Orders with zero remaining quantity"},
    {"Check": "Timed out orders", "Result": timed_out_orders, "Meaning": "Should be zero for a clean run"},
    {"Check": "Total filled quantity", "Result": total_filled, "Meaning": "Aggregate filled quantity across parent orders"},
    {"Check": "Total remaining quantity", "Result": total_remaining, "Meaning": "Residual quantity left after routing"},
    {"Check": "Venues used", "Result": venue_count, "Meaning": "Distinct venues in child fill evidence"},
]
result_list = pd.DataFrame(result_rows)

manifest = frame_from_rows([{
    "scenarioId": scenario_id,
    "seed": seed,
    "ticks": ticks,
    "resetMode": reset_mode,
    "startedAtUtc": started_at.strftime("%Y-%m-%d %H:%M:%S"),
    "completedAtUtc": completed_at.strftime("%Y-%m-%d %H:%M:%S"),
    "maxRouteAttempts": max_route_attempts,
    "routeTimeoutMillis": route_timeout_millis,
    "parentOrderCount": order_count,
}])

what_happened = (
    f"The notebook reset scenario '{scenario_id}' with seed {seed} and ran {ticks} ticks using {reset_mode}. "
    f"It submitted {order_count} parent order(s), produced {child_order_count} child order(s) over {route_attempts} route attempt(s), "
    f"filled {total_filled} units, and left {total_remaining} units remaining. "
    f"The run status is {outcome_label.lower()}."
)

display_styles()
display_header(
    "Scenario Execution Report - Results",
    f"{scenario_id} | seed {seed} | reset {reset_mode}",
    completed_at.strftime("%Y-%m-%d %H:%M:%S"),
)
display(HTML(f"<div class='sor-report'>{badge(outcome_label, outcome_tone)}</div>"))
display_kpis([
    ("Run outcome", outcome_label),
    ("Replay safe", format_bool(replay_safe)),
    ("Filled qty", total_filled),
    ("Remaining qty", total_remaining),
    ("Child orders", child_order_count),
    ("Venues used", venue_count),
])
display_note(what_happened)

display_table("1. Run Manifest", manifest, "Exactly what was run and when.")
display_table("2. Result List", result_list, "Plain-language checklist of the important results.")
order_columns = [
    "scenarioId", "parentOrderId", "clientOrderRef", "atTick", "submitMode",
    "filledQty", "remainingQty", "statusName", "childOrderCount", "routeAttempts", "timedOut"
]
display_table(
    "3. Parent Order Results",
    parent_order_results[[column for column in order_columns if column in parent_order_results.columns]],
    "One row per submitted parent order.",
)
fill_columns = [
    "clientOrderRef", "routeAttempt", "instrumentSymbol", "venueId", "venueName",
    "sideName", "childQty", "filledQty", "priceTicks", "notionalTicks", "policyVersion", "policyHash64"
]
display_table(
    "4. Venue Fill Breakdown",
    fills[[column for column in fill_columns if column in fills.columns]] if not fills.empty else fills,
    "Venue-level evidence for child orders and fills.",
)
display_table("5. Scenario Summary", summary_frame, "Deterministic replay summary returned by the scenario service.")
display_table("6. Reset Evidence", reset_frame, "What the reset did before the scenario run.")
display_table("7. Run Response Evidence", run_frame, "Top-level run response excluding nested parent order details shown above.")
event_columns = ["sequence", "eventId", "timestampNanos", "event", "componentId", "eventType", "correlationId", "message"]
display_table(
    "8. Lifecycle Event Log",
    events[[column for column in event_columns if column in events.columns]],
    "Control-plane and lifecycle events captured during the run.",
)
